# **Does Housing Supply Constrain Home Prices? Evidence from U.S. Metro Areas**

### ECON 320 Lab — Final Project
**Student names:** Tessa Butler, Siya Kumar, Ania Ting, Naman Keswani, Shunji Lewandowski  
**Lab Section:** [#]  

---
## Abstract
We study whether new residential construction activity is associated with lower home prices across U.S. Metropolitan Statistical Areas (MSAs) using a **pooled cross-section** of 922 MSAs over 2019–2024. The unit of observation is an MSA-year pair ($N \approx 5{,}500$). The dependent variable is the log of the FHFA All-Transactions House Price Index. The main regressor is log building permits (Census BPS). The baseline specification is:

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \sum_{t} \delta_t \cdot \text{Year}_t + \gamma\,\text{Metro}_i + u_{it}$$

We find that **[main result — complete after running regressions]**. Adding controls for median income and population **(Model 2)** shifts $\hat{\beta}_1$ **[direction]**, consistent with **upward OVB** from demand-side factors. Results are **[robust/not robust]** to using annual HPI percent change as an alternative DV. We test for heteroskedasticity using the Breusch–Pagan and White tests, and check multicollinearity via VIF. Our findings suggest **[policy implication]**.

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats import diagnostic as sm_diagnostic
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.random.seed(320)

---
## 1. Introduction

- **Research question:** Is greater housing supply — measured by new residential building permits — associated with lower home prices across U.S. metropolitan areas?

- **Why it matters:** The U.S. housing supply gap widened to an estimated 4.03 million homes in 2025 (Realtor.com, 2026). Over 42 million Americans spend more than 30% of income on housing. Standard supply-and-demand theory predicts that constrained supply pushes prices above competitive equilibrium: when supply is low relative to demand, prices rise. This is the core economic mechanism we test empirically.

- **Preview of approach:** We use FHFA house price index data (2019–2024) merged with Census building permit data across ~920 MSAs. We estimate a **log-log OLS** model where the coefficient on log permits is the **price-supply elasticity** — the percent change in home prices associated with a 1% increase in permit activity. We control for year effects (pooled cross-section), metropolitan classification, and (when available) income and population. We explicitly analyze OVB direction, test for multicollinearity via VIF, and check heteroskedasticity using Breusch–Pagan and White tests (Week 10). All standard errors use **HC0** (White/Huber robust) to address heteroskedasticity.

---
## 2. Data and Summary Statistics

**Data sources & unit of observation:**

Our unit of observation is an **MSA-year pair**. We pool 922 MSAs across 6 years (2019–2024) for approximately 5,500 observations — a **pooled cross-section** (Wooldridge Ch. 1).

| Source | Variable | Description |
|--------|----------|-------------|
| FHFA | `HPI` | All-Transactions House Price Index (annual, CBSA). Repeat-sales index using Fannie Mae/Freddie Mac + FHA + county recorder data. Measures cumulative nominal appreciation (base year varies). |
| FHFA | `Ann_Chg_Pct` | Year-over-year percent change in HPI. Used in robustness check. |
| Census BPS | `total_permits` | Total new privately-owned residential units authorized by building permit (CBSA monthly, Jan 2026 snapshot). Cross-sectional supply proxy — MSA permit rankings are highly persistent over time. |
| Census BPS | `metro_dummy` | = 1 if Metropolitan Statistical Area, = 0 if Micropolitan. Captures urban density and demand scale (Week 5: dummy variables). |

**Variable definitions:**

| Role | Variable | Construction | Expected sign |
|------|----------|--------------|---------------|
| DV ($y$) | `log_hpi` | $\ln(\text{HPI}_{it})$ | — |
| Main regressor ($x$) | `log_permits` | $\ln(\text{Total permits}_i + 1)$ | Negative ($-$) |
| Controls | `Year_2020` … `Year_2024` | Year dummies (base = 2019) | Positive (price appreciation over time) |
| Control | `metro_dummy` | = 1 if Metro, 0 if Micro | Positive (metros have higher prices) |

**Why log-log?** The natural log transformation (Wooldridge, Appendix A.4) serves two purposes:
1. It reduces right-skew in both HPI and permits (large MSAs dominate in levels).
2. The coefficient $\beta_1$ is directly interpretable as an **elasticity**: a 1% increase in permits is associated with a $\beta_1$% change in HPI.

**Why `+1` before log?** Some MSAs have zero permits in the monthly snapshot. Since $\ln(0)$ is undefined, we add 1 before taking logs. This preserves zero-permit MSAs in the sample. We verify robustness by dropping zero-permit MSAs instead (Section 5).

**Limitation of permits data:** We use a single monthly snapshot (Jan 2026) rather than annual totals. This introduces measurement noise but does not bias the cross-sectional ranking of MSAs by supply activity, since permit levels are highly persistent across metros.

### 2.1 Load and prepare data

In [ ]:
hpi_raw = pd.read_excel('data/hpi_at_cbsa.xlsx', skiprows=5, header=None)
hpi_raw.columns = ['CBSA', 'Name', 'Year', 'Ann_Chg_Pct', 'HPI', 'HPI_1990', 'HPI_2000']

for col in ['CBSA', 'Year', 'Ann_Chg_Pct', 'HPI']:
    hpi_raw[col] = pd.to_numeric(hpi_raw[col], errors='coerce')

hpi = hpi_raw[
    (hpi_raw['CBSA'] >= 10000) &
    (hpi_raw['Year'].between(2019, 2024))
][['CBSA', 'Name', 'Year', 'HPI', 'Ann_Chg_Pct']].copy()

hpi['CBSA'] = hpi['CBSA'].astype(int)
hpi = hpi.dropna(subset=['HPI']).reset_index(drop=True)
print(f'FHFA HPI loaded: {len(hpi)} MSA-year observations')
print(f'Years: {sorted(hpi["Year"].unique())}')
print(f'Unique MSAs: {hpi["CBSA"].nunique()}')
hpi.head()

In [ ]:
perm_raw = pd.read_excel('data/cbsamonthly_202601.xls', engine='xlrd', skiprows=6, header=None)
data_rows = perm_raw.iloc[2:].reset_index(drop=True)

perm = pd.DataFrame({
    'CBSA': pd.to_numeric(data_rows[1], errors='coerce'),
    'metro_code': pd.to_numeric(data_rows[3], errors='coerce'),
    'total_permits': pd.to_numeric(data_rows[4], errors='coerce')
})
perm = perm.dropna(subset=['CBSA', 'total_permits'])
perm['CBSA'] = perm['CBSA'].astype(int)

perm['metro_dummy'] = (perm['metro_code'].isin([2, 4])).astype(int)

print(f'Permits loaded: {len(perm)} CBSAs')
print(f'Metro/Micro split: {perm["metro_dummy"].value_counts().to_dict()}')
perm.head()

> **Reading the code above:** The `metro_code` column from Census BPS uses 2 = Metropolitan, 4 = Metropolitan Division, 5 = Micropolitan. We create a binary dummy variable `metro_dummy` (Week 5: dummy variables) where 1 = Metro and 0 = Micro. This is our qualitative regressor — it captures the structural difference in price levels between larger metros and smaller micropolitan areas.

In [ ]:
df = hpi.merge(perm[['CBSA', 'total_permits', 'metro_dummy']], on='CBSA', how='inner')
print(f'Merged: {len(df)} MSA-year observations ({df["CBSA"].nunique()} unique MSAs)')
df.head()

In [ ]:
df['log_hpi']     = np.log(df['HPI'])
df['log_permits'] = np.log(df['total_permits'] + 1)

year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
df = pd.concat([df, year_dummies], axis=1)

print(f'Observations: {len(df)}')
print(f'Zero-permit MSAs: {(df["total_permits"] == 0).sum()} ({(df["total_permits"]==0).mean()*100:.1f}%)')
print(f'Year dummies created: {[c for c in year_dummies.columns]}')
df[['CBSA','Name','Year','HPI','log_hpi','total_permits','log_permits','metro_dummy']].head(10)

> **Why year dummies?** In a pooled cross-section (Wooldridge Ch. 1 §3.iii), observations from different years are pooled together, but aggregate conditions (mortgage rates, national demand, COVID shock) differ across years. Year dummies absorb these **aggregate time effects** — they capture anything that changed nationally between 2019 and 2024. The omitted base year is 2019.

> **Interpretation:** The coefficient on `Year_2021`, for example, tells us how much higher (or lower) log HPI was in 2021 compared to 2019, holding permits and metro status constant.

### 2.2 Descriptive statistics *(table + figure)*

In [ ]:
desc_vars = ['HPI', 'Ann_Chg_Pct', 'total_permits', 'log_hpi', 'log_permits', 'metro_dummy']
desc = df[desc_vars].describe().T[['count','mean','std','min','25%','50%','75%','max']].round(3)
print(desc.to_string())

> **Observation:** [After running, comment on the mean/median of HPI, the share of metro vs micro, and the spread of permits. Note any skewness in the raw variables that motivates the log transform.]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['HPI'], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('FHFA HPI (Annual)')
axes[0].set_ylabel('Count')
axes[0].set_title('Fig 1a: Distribution of HPI (levels)')

axes[1].hist(df['log_hpi'], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Log(HPI)')
axes[1].set_title('Fig 1b: Distribution of Log(HPI)')

axes[2].hist(df['log_permits'], bins=50, color='coral', edgecolor='white', linewidth=0.3)
axes[2].set_xlabel('Log(Permits + 1)')
axes[2].set_title('Fig 1c: Distribution of Log(Permits)')

plt.tight_layout()
plt.savefig('fig1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

> **Observation:** The log transform substantially reduces right-skew in both HPI and permits, making the distributions closer to normal. This supports using a log-log specification.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

slope, intercept, r, pv, _ = stats.linregress(df['log_permits'], df['log_hpi'])
x_line = np.linspace(df['log_permits'].min(), df['log_permits'].max(), 200)

axes[0].scatter(df['log_permits'], df['log_hpi'], alpha=0.15, s=10,
               color='steelblue', edgecolors='none')
axes[0].plot(x_line, slope * x_line + intercept, color='firebrick', lw=2)
axes[0].set_xlabel('Log(Permits + 1)')
axes[0].set_ylabel('Log(FHFA HPI)')
axes[0].set_title('Fig 2: Log-Log Scatter — Supply vs. Prices')
axes[0].text(0.04, 0.93, f'r = {r:.3f}, p = {pv:.2e}',
            transform=axes[0].transAxes, fontsize=9)

metro_means = df.groupby('metro_dummy')['log_hpi'].mean()
axes[1].bar(['Micropolitan\n(metro=0)', 'Metropolitan\n(metro=1)'],
           metro_means.values, color=['coral','steelblue'], edgecolor='k', linewidth=0.3)
axes[1].set_ylabel('Mean Log(HPI)')
axes[1].set_title('Fig 2b: Mean Log HPI by Metro Status')
for i, v in enumerate(metro_means.values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('fig2_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

> **Observation:** The scatter plot shows a [positive/negative] raw correlation between log permits and log HPI (r = [value]). [If positive: this is counterintuitive under supply-demand theory and suggests OVB — high-demand metros attract both more construction AND higher prices. Adding controls should reveal the true negative supply effect.]  
> The bar chart confirms that metropolitan areas have substantially higher average HPI than micropolitan areas, motivating inclusion of the metro dummy.

---
## 3. Empirical Strategy

We estimate three nested OLS specifications to progressively address omitted variable bias:

**Model 1 — Bivariate (OVB benchmark, "short" regression):**

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + u_{it}$$

**Model 2 — Year dummies + metro dummy (pooled cross-section):**

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \gamma\,\text{Metro}_i + \sum_{t=2020}^{2024} \delta_t \cdot \mathbf{1}\{\text{Year}=t\} + u_{it}$$

**Model 3 — Full specification (with ACS controls, if available):**

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \gamma\,\text{Metro}_i + \beta_2 \ln(\text{Income}_i) + \beta_3 \ln(\text{Pop}_i) + \sum_{t} \delta_t \cdot \text{Year}_t + u_{it}$$

---

### Interpretation of $\beta_1$

Since both sides are in logs, $\beta_1$ is the **price-supply elasticity** (Wooldridge Appendix A.4):

> *"A 1% increase in building permits is associated with a $\beta_1$% change in the house price index, holding year, metro status, and other controls constant."*

Supply-and-demand theory predicts **$\beta_1 < 0$**: more new construction → greater supply → lower prices.

---

### Identification concerns

**Omitted Variable Bias (OVB) — Week 7:**

The bivariate estimate (Model 1) is likely **upward biased** (toward zero or positive). Here is the OVB logic:

1. **Omitted variable:** Demand factors (income, population) are missing from Model 1.
2. **Correlation with $x$:** High-income, high-population metros issue more permits (developers respond to demand). So $\text{Corr}(\text{Permits}, \text{Income}) > 0$.
3. **Effect on $y$:** Higher income/population → higher housing demand → higher prices. So the true coefficient on Income in the full model is **positive**.
4. **Direction of bias:** $\text{Bias} = (\text{effect of Income on } y) \times (\text{effect of Income on } x) > 0$. This pushes $\hat{\beta}_1$ **upward** (less negative or even positive).

*Prediction:* When we add controls (Model 2 → Model 3), $\hat{\beta}_1$ should become **more negative**, moving toward the true supply effect.

**Multicollinearity — Week 6:**

Log permits and the metro dummy may be positively correlated (metros permit more). This does not bias OLS (it remains unbiased under Gauss-Markov — Week 3), but inflates standard errors. We check using VIF (Week 6: VIF $> 10$ is concerning).

**Heteroskedasticity — Week 10:**

Residual variance likely differs across MSA sizes (large metros have more volatile prices). Under heteroskedasticity, classical SEs are invalid (Wooldridge Ch. 8). We:
1. Use **HC0 robust standard errors** throughout (`.fit(cov_type="HC0")` — Week 10).
2. Run **Breusch–Pagan** and **White** tests to formally detect heteroskedasticity.

**Estimation:** OLS with HC0 robust SEs (Wooldridge Ch. 8).

### 3.1 Estimation details

In [ ]:
y = df['log_hpi']

X1 = sm.add_constant(df['log_permits'])
model1 = sm.OLS(y, X1).fit(cov_type='HC0')

print('=== Model 1: Bivariate (Short Regression) ===')
print(model1.summary())

> **Reading Model 1:** The coefficient on `log_permits` is $\hat{\beta}_1 = $ [value]. This means a 1% increase in permits is associated with a [value]% change in HPI. [If positive: this is consistent with OVB — demand-rich metros build more AND have higher prices. The bivariate regression conflates supply and demand.] $R^2 = $ [value], meaning permits alone explain [X]% of cross-MSA variation in log prices. The $t$-statistic is [value] with $p$-value [value], so the coefficient is [significant/not significant] at the 5% level.

In [ ]:
year_cols = [c for c in df.columns if c.startswith('Year_')]
X2_vars = ['log_permits', 'metro_dummy'] + year_cols
X2 = sm.add_constant(df[X2_vars])
model2 = sm.OLS(y, X2).fit(cov_type='HC0')

print('=== Model 2: + Year Dummies + Metro Dummy ===')
print(model2.summary())

> **Reading Model 2:** After controlling for year effects and metro status:
> - $\hat{\beta}_1$ changed from [M1 value] to [M2 value]. This [increase/decrease] is [consistent/inconsistent] with our OVB prediction.
> - The `metro_dummy` coefficient tells us metros have [X]% higher HPI than micropolitans, holding permits and year constant (interpretation of dummy coefficient in a log model: approximately $100 \times \hat{\gamma}$% — Week 5).
> - Year dummies show the trajectory of national house price appreciation since 2019. The large positive coefficient on Year_2021/2022 likely captures the pandemic housing boom.
> - $R^2$ improved from [M1] to [M2], meaning these controls explain additional variation.

In [ ]:
comparison = pd.DataFrame({
    'Model 1 (Short)': [model1.params.get('log_permits', np.nan),
                        model1.bse.get('log_permits', np.nan),
                        model1.pvalues.get('log_permits', np.nan),
                        model1.rsquared,
                        int(model1.nobs)],
    'Model 2 (Long)': [model2.params.get('log_permits', np.nan),
                       model2.bse.get('log_permits', np.nan),
                       model2.pvalues.get('log_permits', np.nan),
                       model2.rsquared,
                       int(model2.nobs)]
}, index=['β₁ (log_permits)', 'SE(β₁) HC0', 'p-value', 'R²', 'N'])

print('=== OVB Comparison: Short vs Long Regression ===')
print(comparison.round(4).to_string())
print(f'\nEmpirical OVB = β₁(short) - β₁(long) = {(model1.params["log_permits"] - model2.params["log_permits"]):.4f}')

> **OVB analysis (Week 7):** The empirical OVB is the difference in $\hat{\beta}_1$ between Model 1 and Model 2. A positive OVB means the short regression overestimates the supply effect (or masks a negative effect). This is [consistent/inconsistent] with our theoretical prediction from Section 3.

### 3.2 Diagnostic awareness

#### Multicollinearity check (VIF) — Week 6

In [ ]:
X_vif = df[X2_vars].dropna()
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(sm.add_constant(X_vif).values, i+1)
            for i in range(X_vif.shape[1])]
}).round(2)

print('=== Variance Inflation Factors ===')
print(vif_data.to_string(index=False))
print(f'\nRule of thumb: VIF > 10 suggests problematic multicollinearity (Week 6).')
print(f'Max VIF: {vif_data["VIF"].max():.2f}')

> **Interpretation:** VIF values [are/are not] above 10, so multicollinearity [is/is not] a concern. Even if some VIFs are elevated, recall from Week 6 that multicollinearity inflates standard errors but does **not** bias the OLS coefficients. OLS remains BLUE under Gauss-Markov (Week 3, Theorem).

#### Heteroskedasticity tests — Week 10

In [ ]:
X2_full = sm.add_constant(df[X2_vars])
ols_for_test = sm.OLS(y, X2_full).fit()

bp_lm, bp_p, _, _ = sm_diagnostic.het_breuschpagan(ols_for_test.resid, X2_full)
white_lm, white_p, _, _ = sm_diagnostic.het_white(ols_for_test.resid, X2_full)

alpha = 0.05
print('=== Heteroskedasticity Tests (Week 10) ===')
print(f'Breusch-Pagan LM = {bp_lm:.3f}, p-value = {bp_p:.4f}')
print(f'White LM         = {white_lm:.3f}, p-value = {white_p:.4f}')
print(f'\nDecision @ {alpha*100:.0f}%: ' + (
    'Reject H₀ of homoskedasticity → variance is NOT constant → use HC0 robust SEs ✓'
    if (bp_p < alpha or white_p < alpha)
    else 'Cannot reject homoskedasticity at 5%'))

> **Interpretation:** The Breusch–Pagan test regresses $\hat{e}_i^2$ on the original regressors and tests whether the coefficients are jointly zero (Week 10). The White test adds squares and cross-products. Both test $H_0$: homoskedasticity. A small $p$-value rejects $H_0$. Since we [do/do not] reject, using HC0 robust SEs is [necessary/precautionary but harmless].

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(model2.fittedvalues, model2.resid, alpha=0.15, s=8,
          color='steelblue', edgecolors='none')
ax.axhline(0, color='firebrick', lw=1.5, ls='--')
ax.set_xlabel('Fitted Values — Log(HPI)')
ax.set_ylabel('OLS Residuals')
ax.set_title('Fig 3: Residuals vs. Fitted Values — Model 2')
plt.tight_layout()
plt.savefig('fig3_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

> **Reading the residual plot (Week 10):** If we see a "funnel" shape (spread increasing with fitted values), that is visual evidence of heteroskedasticity. [Describe what you see.] This is why we use HC0 SEs.

#### F-test for joint significance of year dummies — Week 9

In [ ]:
X_r = sm.add_constant(df[['log_permits', 'metro_dummy']])
model_restricted = sm.OLS(y, X_r).fit()

SSR_R  = model_restricted.ssr
SSR_UR = model2.ssr
q = len(year_cols)
n_obs = int(model2.nobs)
k_ur  = X2.shape[1]

F_stat = ((SSR_R - SSR_UR) / q) / (SSR_UR / (n_obs - k_ur))
p_F = 1 - stats.f.cdf(F_stat, q, n_obs - k_ur)

print('=== F-test: Joint Significance of Year Dummies (Week 9) ===')
print(f'H₀: δ_2020 = δ_2021 = δ_2022 = δ_2023 = δ_2024 = 0')
print(f'H₁: At least one δ ≠ 0')
print(f'\nSSR(Restricted)   = {SSR_R:.3f}')
print(f'SSR(Unrestricted) = {SSR_UR:.3f}')
print(f'q = {q} restrictions, df = {n_obs - k_ur}')
print(f'F-statistic = {F_stat:.3f}')
print(f'p-value     = {p_F:.2e}')
print(f'\nDecision @ 5%: ' + ('Reject H₀ — year dummies are jointly significant' if p_F < 0.05 else 'Cannot reject H₀'))

> **Interpretation (Week 9):** The $F$-test tests whether the year dummies are *jointly* significant — i.e., whether aggregate time trends matter after controlling for permits and metro status. We compute $F = \frac{(SSR_R - SSR_{UR})/q}{SSR_{UR}/(n-k-1)}$ manually (as in Lab 9). [If significant: year dummies capture important aggregate price trends (pandemic boom, interest rate changes) and should remain in the model.]

---
## 4. Results

**Main result table and interpretation:**

In [ ]:
results_table = pd.DataFrame({
    'Model 1 (Bivariate)': model1.params.round(4),
    'SE (M1)': model1.bse.round(4),
    'Model 2 (Full)': model2.params.round(4),
    'SE (M2)': model2.bse.round(4)
})
results_table.loc['R²'] = [model1.rsquared, '', model2.rsquared, '']
results_table.loc['N']  = [int(model1.nobs), '', int(model2.nobs), '']
results_table = results_table.round(4)
print('=== Regression Results ===')
print(results_table.to_string())

> **Reading the table:**
> - **$\hat{\beta}_1$ (log_permits):** In Model 2, a 1% increase in building permits is associated with a [β₁]% change in HPI. [Is it negative as theory predicts?]
> - **metro_dummy:** Metropolitan areas have approximately $100 \times \hat{\gamma}$% = [value]% higher HPI than micropolitan areas, holding permits and year constant.
> - **Year dummies:** [Describe the trajectory — e.g., prices were X% higher in 2022 vs 2019.]
> - **$R^2$:** The full model explains [value]% of cross-MSA-year variation in log prices.
> - **Statistical significance:** $\hat{\beta}_1$ has a $t$-statistic of [value] and $p$-value of [value], so it is [statistically significant / not significant] at the 5% level.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sm.qqplot(model2.resid, line='s', ax=ax, alpha=0.2, markersize=3)
ax.set_title('Fig 4: Q-Q Plot of Residuals — Model 2')
plt.tight_layout()
plt.savefig('fig4_qq.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Robustness and Limitations

We perform three robustness checks to assess the stability of our main result.

**Robustness 1:** Alternative DV — annual HPI percent change instead of log level.

In [ ]:
y_rob1 = df['Ann_Chg_Pct'].dropna()
X_rob1 = sm.add_constant(df.loc[y_rob1.index, X2_vars])
rob1 = sm.OLS(y_rob1, X_rob1).fit(cov_type='HC0')
print('=== Robustness 1: DV = Annual HPI % Change ===')
print(f'β₁ (log_permits) = {rob1.params["log_permits"]:.4f}')
print(f'SE               = {rob1.bse["log_permits"]:.4f}')
print(f'p-value           = {rob1.pvalues["log_permits"]:.4f}')
print(f'R²                = {rob1.rsquared:.4f}')

**Robustness 2:** Drop zero-permit MSAs (instead of using log(permits+1)).

In [ ]:
df_nz = df[df['total_permits'] > 0].copy()
df_nz['log_permits_strict'] = np.log(df_nz['total_permits'])
y_nz = df_nz['log_hpi']
X_nz_vars = ['log_permits_strict', 'metro_dummy'] + year_cols
X_nz = sm.add_constant(df_nz[X_nz_vars])
rob2 = sm.OLS(y_nz, X_nz).fit(cov_type='HC0')
print(f'=== Robustness 2: Drop zero-permit MSAs (N={len(df_nz)}) ===')
print(f'β₁ (log_permits) = {rob2.params["log_permits_strict"]:.4f}')
print(f'p-value           = {rob2.pvalues["log_permits_strict"]:.4f}')
print(f'Compare to Model 2: β₁ = {model2.params["log_permits"]:.4f}')

**Robustness 3:** Trim extreme outliers (top/bottom 2.5% of permits).

In [ ]:
lo, hi = df['log_permits'].quantile(0.025), df['log_permits'].quantile(0.975)
df_trim = df[(df['log_permits'] >= lo) & (df['log_permits'] <= hi)]
y_t = df_trim['log_hpi']
X_t = sm.add_constant(df_trim[X2_vars])
rob3 = sm.OLS(y_t, X_t).fit(cov_type='HC0')
print(f'=== Robustness 3: Trimmed sample (N={len(df_trim)}) ===')
print(f'β₁ (log_permits) = {rob3.params["log_permits"]:.4f}')
print(f'p-value           = {rob3.pvalues["log_permits"]:.4f}')
print(f'Compare to Model 2: β₁ = {model2.params["log_permits"]:.4f}')

In [ ]:
rob_summary = pd.DataFrame({
    'Model': ['Model 2 (Main)', 'Rob 1: Ann. Chg DV', 'Rob 2: Drop zeros', 'Rob 3: Trim 5%'],
    'β₁': [model2.params['log_permits'], rob1.params['log_permits'],
           rob2.params['log_permits_strict'], rob3.params['log_permits']],
    'p-value': [model2.pvalues['log_permits'], rob1.pvalues['log_permits'],
               rob2.pvalues['log_permits_strict'], rob3.pvalues['log_permits']],
    'R²': [model2.rsquared, rob1.rsquared, rob2.rsquared, rob3.rsquared],
    'N': [int(model2.nobs), int(rob1.nobs), int(rob2.nobs), int(rob3.nobs)]
}).round(4)
print('=== Robustness Summary ===')
print(rob_summary.to_string(index=False))

**Limitations:**

1. **Reverse causality:** Higher prices may *attract* more construction (developers respond to profit opportunities). Our OLS estimates capture an association, not a clean causal effect. An instrumental variable (Week 11) — such as geographic constraints on building (steep slopes, water boundaries) or zoning strictness indices — would address this. We do not pursue IV here.

2. **Timing mismatch:** Permits are from a single monthly snapshot (Jan 2026), while HPI covers 2019–2024. Annual permit totals matched to each year would be ideal for a proper pooled cross-section.

3. **Missing demand-side controls:** Without MSA-level income and population (ACS data needs re-downloading at MSA level), Model 2 still suffers from OVB. The metro dummy partially proxies for demand scale but is coarse.

4. **Not a panel:** Although we observe the same MSAs over time, we treat this as pooled cross-sections because our key regressor (permits) does not vary over time in our data. A true panel with year-varying permits would allow fixed-effects estimation to remove time-invariant MSA confounders.

---
## 6. Conclusion

Using a pooled cross-section of ~920 U.S. MSAs over 2019–2024 ($N \approx 5{,}500$), we find that building permit activity is [positively/negatively] associated with home price levels. The log-log elasticity estimate in our preferred specification (Model 2) is $\hat{\beta}_1 = $ [value], meaning a 1% increase in permitted units is associated with a [value]% [increase/decrease] in the FHFA House Price Index, controlling for year effects and metro classification.

The shift in $\hat{\beta}_1$ from Model 1 to Model 2 is consistent with upward OVB from demand-side factors, as predicted by economic theory. [If β₁ is still positive even in Model 2: this suggests demand-side OVB dominates and additional controls (income, population) are needed to isolate the true supply effect. If β₁ is negative: the supply effect is detectable even in this cross-sectional setting.]

**Policy implication:** [If negative: Expanding permitting in supply-constrained metros could moderate price appreciation. If positive: The demand channel dominates — policy must address demand-side affordability (subsidies, mortgage access) alongside supply expansion.]

---
## References
- **FHFA House Price Index:** Federal Housing Finance Agency. *FHFA HPI All-Transactions CBSA Annual Data.* https://www.fhfa.gov/data/hpi/datasets
- **Census BPS:** U.S. Census Bureau. *New Residential Building Permits, CBSA Monthly.* https://www.census.gov/construction/bps
- **Realtor.com.** (2026). *2026 Housing Supply Gap Report.*
- Glaeser, E. L., Gyourko, J., & Saks, R. E. (2005). Why have housing prices gone up? *American Economic Review, 95*(2), 329–333.
- Wooldridge, J. M. (2019). *Introductory Econometrics: A Modern Approach* (7th ed.). Cengage.

---